# Sampling and Temperature — companion notebook

Watch diversity and tail-reach move as you turn the knobs, and see where temperature and truncation part company.

## 1. One distribution, four samplers

In [ ]:
import math, random
from collections import Counter

DIST = {"mat":0.45,"rug":0.25,"floor":0.15,"carpet":0.08,
        "table":0.04,"moon":0.02,"xylophone":0.01}
NONSENSE = {"moon", "xylophone"}

def rescale(d, T):
    if T <= 0: return {max(d, key=d.get): 1.0}
    lg = {t: math.log(p)/T for t, p in d.items()}
    m = max(lg.values()); e = {t: math.exp(v-m) for t, v in lg.items()}
    z = sum(e.values()); return {t: v/z for t, v in e.items()}

def top_p(d, p):
    kept, run = {}, 0.0
    for t, q in sorted(d.items(), key=lambda kv: -kv[1]):
        kept[t] = q; run += q
        if run >= p: break
    z = sum(kept.values()); return {t: q/z for t, q in kept.items()}

def sample(d, rng):
    r, acc = rng.random(), 0.0
    for t, p in d.items():
        acc += p
        if r <= acc: return t
    return next(reversed(d))

def draw(d, n=2000, seed=0):
    rng = random.Random(seed)          # ONE rng for the run, not one per draw
    return Counter(sample(d, rng) for _ in range(n))

print(draw(DIST).most_common())

## 2. The curves cross

Diversity and nonsense both rise with temperature — that's the trap. Truncation breaks the link: variety without the tail.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

temps = [i/10 for i in range(1, 26)]
distinct, junk = [], []
for T in temps:
    got = draw(rescale(DIST, T))
    distinct.append(len(got))
    junk.append(sum(got[t] for t in NONSENSE) / 2000 * 100)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(temps, distinct, label="distinct tokens seen")
ax.set_xlabel("temperature"); ax.set_ylabel("distinct tokens", color="tab:blue")
ax.grid(alpha=.3)
ax2 = ax.twinx()
ax2.plot(temps, junk, color="tab:red", label="% nonsense")
ax2.set_ylabel("% draws that were nonsense", color="tab:red")

nuc = draw(top_p(DIST, 0.85))
ax.scatter([1.0], [len(nuc)], marker="*", s=260, color="green", zorder=5)
ax.annotate("top_p=0.85\n(0% nonsense)", (1.0, len(nuc)),
            textcoords="offset points", xytext=(12, -22), color="green")
ax.set_title("Temperature couples diversity to nonsense. Truncation does not.")
plt.tight_layout(); plt.show()

print(f"top_p=0.85 -> {len(nuc)} distinct, "
      f"{sum(nuc[t] for t in NONSENSE)/2000*100:.1f}% nonsense")

## 3. Find the setting that meets a requirement

State the requirement first — *at least 3 distinct options, zero nonsense* — then search for settings that satisfy it, rather than tuning by feel.

In [ ]:
print(f"{'setting':<26} {'distinct':>9} {'% nonsense':>11}  meets spec?")
cands = [(f"T={T}", rescale(DIST, T)) for T in (0.5, 1.0, 1.5, 2.0)]
cands += [(f"T=1.0 + top_p={p}", top_p(DIST, p)) for p in (0.75, 0.85, 0.95)]
for label, d in cands:
    got = draw(d)
    n_junk = sum(got[t] for t in NONSENSE)
    ok = "YES" if len(got) >= 3 and n_junk == 0 else "no"
    print(f"{label:<26} {len(got):>9} {n_junk/2000*100:>10.1f}%  {ok}")
print("\nNo temperature alone satisfies it. Truncation is not optional here.")

## 4. Your turn

1. Replace `DIST` with real logprobs from your provider (most return them).
2. Redefine `NONSENSE` as whatever your parser rejects.
3. Write your requirement as a predicate and search the settings grid for it — that is a tuning method, and 'it feels better' is not.